In [ ]:
# ============================================================
# INTEGRACIÓN CON GOOGLE SHEETS
# ============================================================
# Extrae las 3 matrices (escudo, espada, descarte) desde una pestaña de
# un Google Sheet, sin necesidad de saber de antemano su posición ni su
# tamaño. Cada bloque se identifica porque su celda superior-izquierda
# contiene el texto "escudo", "espada" o "descarte".
#
# IMPORTANTE (semántica de las matrices):
#   - "descarte" es la matriz de payout BASE (los valores reales).
#   - "escudo" y "espada" son MODIFICADORES que se aplican sobre la de
#     descarte: M_escudo_final = M_descarte + Mod_escudo
#              M_espada_final  = M_descarte + Mod_espada
#     (modo "aditivo", por defecto). También se soporta modo
#     "multiplicativo" si tus modificadores están pensados como
#     factores en vez de sumandos.
#
# Formato esperado de cada bloque en la hoja (ejemplo con N=3):
#
#     escudo    B1   B2   B3
#     A1         5    3    5
#     A2         4    4    4
#     A3         5    3    5
#
# Los 3 bloques pueden estar en cualquier posición de la hoja, en
# cualquier orden, con cualquier separación entre ellos (filas/columnas
# en blanco), siempre que tengan el mismo número de filas y columnas.

import numpy as np

ROLES_VALIDOS = {"escudo", "espada", "descarte"}


def _normalizar(texto):
    t = str(texto).strip().lower()
    t = t.rstrip(":").strip()
    return t


def _detectar_bloques(datos):
    """
    `datos`: lista de listas (filas x columnas) de strings, tal como
    devuelve worksheet.get_all_values() de gspread.

    Busca celdas cuyo contenido normalizado sea 'escudo', 'espada' o
    'descarte' y, a partir de cada una, extrae las cabeceras (a la
    derecha y hacia abajo) y la matriz numérica que forman.

    Devuelve un dict {rol: {"matriz": np.array, "nombres_fila": [...],
                             "nombres_columna": [...], "posicion": (fila, col)}}
    """
    n_filas_total = len(datos)
    bloques = {}

    for r in range(n_filas_total):
        fila = datos[r]
        for c in range(len(fila)):
            valor = _normalizar(fila[c])
            if valor not in ROLES_VALIDOS:
                continue
            if valor in bloques:
                raise ValueError(f"El rol '{valor}' aparece más de una vez en la hoja "
                                  f"(ya se había encontrado en {bloques[valor]['posicion']}).")

            # Cabeceras de columna: hacia la derecha, misma fila
            nombres_columna = []
            cc = c + 1
            while cc < len(fila) and fila[cc].strip() != "":
                nombres_columna.append(fila[cc].strip())
                cc += 1

            # Cabeceras de fila: hacia abajo, misma columna
            nombres_fila = []
            rr = r + 1
            while rr < n_filas_total and c < len(datos[rr]) and datos[rr][c].strip() != "":
                nombres_fila.append(datos[rr][c].strip())
                rr += 1

            n = len(nombres_columna)
            if n == 0 or len(nombres_fila) != n:
                raise ValueError(
                    f"Bloque '{valor}' en fila {r + 1}, columna {c + 1}: "
                    f"cabeceras de fila ({len(nombres_fila)}) y de columna ({n}) "
                    f"no coinciden o están vacías. Revisa el formato en la hoja."
                )

            matriz = np.zeros((n, n))
            for i in range(n):
                fila_datos = datos[r + 1 + i] if (r + 1 + i) < n_filas_total else []
                for j in range(n):
                    col_idx = c + 1 + j
                    texto = fila_datos[col_idx] if col_idx < len(fila_datos) else ""
                    texto = texto.strip().replace(",", ".")
                    if texto == "":
                        print(f"⚠️  Bloque '{valor}': celda vacía para fila '{nombres_fila[i]}' "
                              f"y columna '{nombres_columna[j]}'. Se imputa el valor 4.")
                        matriz[i, j] = 4.0
                        continue
                    try:
                        matriz[i, j] = float(texto)
                    except ValueError:
                        raise ValueError(
                            f"Bloque '{valor}': valor no numérico '{texto}' en fila "
                            f"{r + 1 + i + 1}, columna {col_idx + 1}."
                        )

            bloques[valor] = {
                "matriz": matriz,
                "nombres_fila": nombres_fila,
                "nombres_columna": nombres_columna,
                "posicion": (r, c),
            }

    return bloques


def construir_matrices_finales(bloques, modo_modificador="aditivo"):
    """
    A partir del dict de bloques detectados, valida que estén los 3 roles,
    que tengan el mismo tamaño, y construye las 3 matrices FINALES de
    payout (M_escudo, M_espada, M_descarte) listas para usar con
    EspadasYEscudo / AsistenteEspadasYEscudo.
    """
    faltantes = ROLES_VALIDOS - set(bloques.keys())
    if faltantes:
        raise ValueError(f"No se han encontrado en la hoja los bloques: {sorted(faltantes)}. "
                          f"Se necesitan los 3: escudo, espada y descarte.")

    n = bloques["descarte"]["matriz"].shape[0]
    for rol in ROLES_VALIDOS:
        if bloques[rol]["matriz"].shape != (n, n):
            raise ValueError(f"El bloque '{rol}' tiene tamaño {bloques[rol]['matriz'].shape}, "
                              f"pero 'descarte' tiene tamaño {(n, n)}. Los 3 deben coincidir.")

    nombres_A = bloques["descarte"]["nombres_fila"]
    nombres_B = bloques["descarte"]["nombres_columna"]

    # Aviso (no bloqueante) si los nombres de fila/columna difieren entre bloques
    for rol in ("escudo", "espada"):
        if bloques[rol]["nombres_fila"] != nombres_A or bloques[rol]["nombres_columna"] != nombres_B:
            print(f"⚠️  Aviso: los nombres de fila/columna del bloque '{rol}' no coinciden "
                  f"exactamente con los de 'descarte'. Se usarán los de 'descarte' para todo, "
                  f"pero revisa que el orden de jugadores sea el mismo en los 3 bloques.")

    base = bloques["descarte"]["matriz"]
    mod_escudo = bloques["escudo"]["matriz"]
    mod_espada = bloques["espada"]["matriz"]

    if modo_modificador == "aditivo":
        M_escudo = base + mod_escudo
        M_espada = base + mod_espada
    elif modo_modificador == "multiplicativo":
        M_escudo = base * mod_escudo
        M_espada = base * mod_espada
    else:
        raise ValueError("modo_modificador debe ser 'aditivo' o 'multiplicativo'")

    M_descarte = base.copy()
    return M_escudo, M_espada, M_descarte, nombres_A, nombres_B


# ------------------------------------------------------------
# Conexión real a Google Sheets (solo funciona dentro de Colab)
# ------------------------------------------------------------
def _conectar_gspread():
    from google.colab import auth
    import gspread
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    return gspread.authorize(creds)


def _abrir_spreadsheet(gc, nombre_o_url):
    import gspread
    nombre_o_url = nombre_o_url.strip()
    if nombre_o_url.startswith("http"):
        return gc.open_by_url(nombre_o_url)
    try:
        return gc.open(nombre_o_url)
    except gspread.SpreadsheetNotFound:
        # también admitimos que el usuario pase directamente la key del sheet
        return gc.open_by_key(nombre_o_url)


def cargar_matrices_desde_sheets(nombre_archivo=None, nombre_pestana=None, modo_modificador="aditivo"):
    """
    Función principal: conecta con Google Sheets, localiza las 3 matrices
    (escudo/espada/descarte) en la pestaña indicada, y devuelve las
    matrices finales listas para usar en el resto del programa.

    - nombre_archivo: nombre (o URL, o key) del Google Sheet. Si es None,
      se pide por input().
    - nombre_pestana: nombre de la pestaña/hoja a usar. Si es None, se
      pide por input(); si se deja en blanco (Enter), se usa la primera.
    - modo_modificador: 'aditivo' (por defecto) o 'multiplicativo'.

    Devuelve: (M_escudo, M_espada, M_descarte, nombres_A, nombres_B)
    """
    if not nombre_archivo:
        nombre_archivo = input("Nombre (o URL) del Google Sheet a buscar: ").strip()

    if nombre_pestana is None:
        resp = input("Nombre de la pestaña a usar [Enter = la primera]: ").strip()
        nombre_pestana = resp if resp != "" else None

    gc = _conectar_gspread()
    sh = _abrir_spreadsheet(gc, nombre_archivo)
    ws = sh.worksheet(nombre_pestana) if nombre_pestana else sh.sheet1

    print(f"\nLeyendo datos de '{sh.title}' / pestaña '{ws.title}'...")
    datos = ws.get_all_values()

    return _procesar_datos(datos, modo_modificador, origen=f"{sh.title} / {ws.title}")


def _procesar_datos(datos, modo_modificador, origen="hoja"):
    bloques = _detectar_bloques(datos)
    M_escudo, M_espada, M_descarte, nombres_A, nombres_B = construir_matrices_finales(
        bloques, modo_modificador)

    print(f"\nMatrices detectadas correctamente en {origen}:")
    for rol in ("escudo", "espada", "descarte"):
        pos = bloques[rol]["posicion"]
        print(f"  - {rol:9s}: esquina en fila {pos[0] + 1}, columna {pos[1] + 1}, "
              f"tamaño {M_descarte.shape[0]}x{M_descarte.shape[1]}")
    print(f"\nJugadores equipo A (filas):    {nombres_A}")
    print(f"Jugadores equipo B (columnas): {nombres_B}")
    print(f"\nModo de combinación: {modo_modificador}")
    print("\nM_descarte (base):")
    print(M_descarte)
    print("\nM_escudo (final = descarte {} escudo):".format("+" if modo_modificador == "aditivo" else "*"))
    print(M_escudo)
    print("\nM_espada (final = descarte {} espada):".format("+" if modo_modificador == "aditivo" else "*"))
    print(M_espada)

    return M_escudo, M_espada, M_descarte, nombres_A, nombres_B


# ============================================================
# EJECUCIÓN
# ============================================================
# QUÉ HACE FALTA PARA CONECTAR CON GOOGLE SHEETS DESDE COLAB:
#
# 1. Este código debe ejecutarse dentro de un notebook de Google Colab
#    (usa `google.colab.auth`, que no existe fuera de Colab).
#
# 2. La librería `gspread` debe estar instalada. Normalmente ya viene
#    preinstalada en Colab; si no, ejecuta en una celda:
#       !pip install gspread google-auth
#
# 3. Al llamar a cargar_matrices_desde_sheets(), se abrirá un popup de
#    autenticación (auth.authenticate_user()) pidiéndote que:
#       - Inicies sesión con la cuenta de Google que tiene acceso al
#         Google Sheet que quieres leer.
#       - Concedas permiso a Colab para acceder a Google Sheets/Drive
#         en tu nombre (solo hace falta hacerlo una vez por sesión).
#
# 4. El Google Sheet debe ser accesible por esa cuenta (ser su propietaria
#    o tenerlo compartido con ella, aunque sea solo con permiso de lectura).
#
# 5. Le puedes pasar el NOMBRE del archivo tal como aparece en Google
#    Drive, o directamente su URL completa, o su ID/key. Si no le pasas
#    nada, te lo pedirá por input().
#
# 6. Le puedes pasar el nombre de la PESTAÑA a usar; si no le pasas nada
#    (o pulsas Enter cuando te lo pida), usará la primera pestaña de la hoja.

# M_escudo, M_espada, M_descarte, nombres_A, nombres_B = cargar_matrices_desde_sheets("input Wahammer pairing")

In [ ]:
# ============================================================
# JUEGO "ESPADAS Y ESCUDO" - Motor de resolución óptima (minimax)
# ============================================================
# Requiere: numpy, scipy (ya vienen preinstalados en Google Colab)

import numpy as np
import itertools
from scipy.optimize import linprog

# ------------------------------------------------------------
# 1. ELIMINACIÓN DE ESTRATEGIAS DOMINADAS (fuerte y débil)
# ------------------------------------------------------------

def _remove_dominated_rows(mat, row_idx):
    """Elimina filas (jugador que MAXIMIZA) dominadas fuerte o débilmente."""
    row_idx = list(row_idx)
    changed = True
    while changed and len(row_idx) > 1:
        changed = False
        for i in row_idx:
            for i2 in row_idx:
                if i == i2:
                    continue
                fila_i = mat[i, :]
                fila_i2 = mat[i2, :]
                # i2 domina a i si siempre es >= (débil) y en algún punto > (fuerte)
                if np.all(fila_i2 >= fila_i - 1e-9) and np.any(fila_i2 > fila_i + 1e-9):
                    row_idx.remove(i)
                    changed = True
                    break
            if changed:
                break
    return row_idx


def _remove_dominated_cols(mat, col_idx):
    """Elimina columnas (jugador que MINIMIZA) dominadas fuerte o débilmente."""
    col_idx = list(col_idx)
    changed = True
    while changed and len(col_idx) > 1:
        changed = False
        for j in col_idx:
            for j2 in col_idx:
                if j == j2:
                    continue
                col_j = mat[:, j]
                col_j2 = mat[:, j2]
                # j2 domina a j si siempre es <= (débil) y en algún punto < (fuerte)
                if np.all(col_j2 <= col_j + 1e-9) and np.any(col_j2 < col_j - 1e-9):
                    col_idx.remove(j)
                    changed = True
                    break
            if changed:
                break
    return col_idx


def reduce_dominance(mat):
    """
    Elimina iterativamente estrategias dominadas (fuerte y débil),
    alternando filas y columnas hasta que no cambie más.
    Devuelve los índices ORIGINALES de filas y columnas supervivientes.
    """
    rows = list(range(mat.shape[0]))
    cols = list(range(mat.shape[1]))
    prev = None
    while prev != (tuple(rows), tuple(cols)):
        prev = (tuple(rows), tuple(cols))
        sub = mat[np.ix_(rows, cols)]
        new_rows_local = _remove_dominated_rows(sub, list(range(len(rows))))
        rows = [rows[i] for i in new_rows_local]
        sub = mat[np.ix_(rows, cols)]
        new_cols_local = _remove_dominated_cols(sub, list(range(len(cols))))
        cols = [cols[j] for j in new_cols_local]
    return rows, cols


# ------------------------------------------------------------
# 2. RESOLUCIÓN DE UN JUEGO MATRICIAL DE SUMA CERO (minimax / LP)
# ------------------------------------------------------------

def solve_matrix_game(mat):
    """
    Resuelve un juego matricial de suma cero (N filas x M columnas).
    Filas = jugador que MAXIMIZA, columnas = jugador que MINIMIZA.

    1) Elimina estrategias dominadas (fuerte y débil).
    2) Resuelve el juego reducido: punto de silla puro si existe,
       si no, estrategia mixta óptima vía programación lineal
       (teorema del minimax).

    Devuelve:
        value        -> valor del juego
        row_strategy -> vector de probabilidades sobre TODAS las filas originales
        col_strategy -> vector de probabilidades sobre TODAS las columnas originales
    """
    mat = np.asarray(mat, dtype=float)
    n, m = mat.shape
    rows, cols = reduce_dominance(mat)
    sub = mat[np.ix_(rows, cols)]

    row_strategy = np.zeros(n)
    col_strategy = np.zeros(m)

    if sub.shape[0] == 1 and sub.shape[1] == 1:
        value = sub[0, 0]
        row_strategy[rows[0]] = 1.0
        col_strategy[cols[0]] = 1.0
        return value, row_strategy, col_strategy

    # ¿Punto de silla puro? (maximin de filas == minimax de columnas)
    row_mins = sub.min(axis=1)
    best_row_local = int(np.argmax(row_mins))
    maximin = row_mins[best_row_local]

    col_maxs = sub.max(axis=0)
    best_col_local = int(np.argmin(col_maxs))
    minimax = col_maxs[best_col_local]

    if np.isclose(maximin, minimax, atol=1e-9):
        row_strategy[rows[best_row_local]] = 1.0
        col_strategy[cols[best_col_local]] = 1.0
        return maximin, row_strategy, col_strategy

    # No hay punto de silla puro -> estrategia mixta óptima vía LP
    r, c = sub.shape
    shift = abs(sub.min()) + 1.0  # valores positivos, por estabilidad numérica
    A = sub + shift

    # --- Estrategia óptima del jugador FILA (maximiza v) ---
    c_lp = np.zeros(r + 1)
    c_lp[-1] = -1  # minimizamos -v  <=>  maximizamos v
    A_ub = np.hstack([-A.T, np.ones((c, 1))])     # v - sum_i p_i*A[i,j] <= 0
    b_ub = np.zeros(c)
    A_eq = np.hstack([np.ones((1, r)), [[0]]])
    b_eq = [1]
    bounds = [(0, 1)] * r + [(None, None)]
    res_row = linprog(c_lp, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                       bounds=bounds, method="highs")
    p = res_row.x[:r]
    value = res_row.x[-1] - shift

    # --- Estrategia óptima del jugador COLUMNA (minimiza v) ---
    c_lp2 = np.zeros(c + 1)
    c_lp2[-1] = 1
    A_ub2 = np.hstack([A, -np.ones((r, 1))])      # sum_j q_j*A[i,j] - v <= 0
    b_ub2 = np.zeros(r)
    A_eq2 = np.hstack([np.ones((1, c)), [[0]]])
    b_eq2 = [1]
    bounds2 = [(0, 1)] * c + [(None, None)]
    res_col = linprog(c_lp2, A_ub=A_ub2, b_ub=b_ub2, A_eq=A_eq2, b_eq=b_eq2,
                       bounds=bounds2, method="highs")
    q = res_col.x[:c]

    for i, ridx in enumerate(rows):
        row_strategy[ridx] = p[i]
    for j, cidx in enumerate(cols):
        col_strategy[cidx] = q[j]

    return value, row_strategy, col_strategy


# ------------------------------------------------------------
# 3. MOTOR DEL JUEGO "ESPADAS Y ESCUDO"
# ------------------------------------------------------------

class EspadasYEscudo:
    """
    Motor de resolución óptima del juego Espadas y Escudo, mediante
    inducción hacia atrás sobre juegos matriciales anidados de suma cero.

    El payout de cada enfrentamiento depende del ROL que tiene el jugador
    del EQUIPO A en ese combate concreto (escudo, espada o descarte), por
    lo que se reciben 3 matrices de payout NxN, una por rol:

    M_escudo[i, j]   -> resultado cuando el jugador i de A actúa de ESCUDO
                         contra el jugador j de B.
    M_espada[i, j]   -> resultado cuando el jugador i de A actúa de ESPADA
                         (aceptada o rechazada) contra el jugador j de B.
    M_descarte[i, j] -> resultado cuando el jugador i de A actúa de DESCARTE
                         contra el jugador j de B.

    En los 3 casos: filas (equipo A) MAXIMIZAN, columnas (equipo B) MINIMIZAN.
    El rol de equipo B no afecta a qué matriz se usa (solo importa el rol
    del jugador de equipo A en ese emparejamiento).
    """

    def __init__(self, M_escudo, M_espada, M_descarte, nombres_A=None, nombres_B=None):
        self.M_escudo = np.asarray(M_escudo, dtype=float)
        self.M_espada = np.asarray(M_espada, dtype=float)
        self.M_descarte = np.asarray(M_descarte, dtype=float)

        for nombre, M in (("M_escudo", self.M_escudo), ("M_espada", self.M_espada),
                          ("M_descarte", self.M_descarte)):
            n, m = M.shape
            assert n == m, f"{nombre} debe ser cuadrada (NxN)."

        shapes = {self.M_escudo.shape, self.M_espada.shape, self.M_descarte.shape}
        assert len(shapes) == 1, "Las 3 matrices (escudo/espada/descarte) deben tener la misma forma NxN."

        self.N = self.M_escudo.shape[0]
        self.nombres_A = nombres_A or [f"A{i}" for i in range(self.N)]
        self.nombres_B = nombres_B or [f"B{i}" for i in range(self.N)]
        self._memo_round = {}

    def payoff(self, a, b, rol):
        """Devuelve el payout de A=a vs B=b según el rol de A: 'escudo', 'espada' o 'descarte'."""
        if rol == "escudo":
            return float(self.M_escudo[a, b])
        elif rol == "espada":
            return float(self.M_espada[a, b])
        elif rol == "descarte":
            return float(self.M_descarte[a, b])
        raise ValueError(f"Rol desconocido: {rol!r} (debe ser 'escudo', 'espada' o 'descarte')")

    # --------------------------------------------------------
    # Etapa 3 (aceptación): dado escudo y espadas ya revelados,
    # cada equipo elige en secreto cuál espada rival aceptar.
    # --------------------------------------------------------
    def _accept_game(self, shieldA, shieldB, swordsA, swordsB, remA, remB):
        k = len(remA)
        filas = list(swordsB)   # elección de A: cuál espada de B acepta
        cols = list(swordsA)    # elección de B: cuál espada de A acepta
        mat = np.zeros((2, 2))

        for i, sb in enumerate(filas):
            for j, sa in enumerate(cols):
                # shieldA actúa de ESCUDO; sa (nuestra espada aceptada) actúa de ESPADA
                inmediato = self.payoff(shieldA, sb, "escudo") + self.payoff(sa, shieldB, "espada")

                remA_next = tuple(x for x in remA if x not in (shieldA, sa))
                remB_next = tuple(x for x in remB if x not in (shieldB, sb))

                if k == 3:
                    # queda 1 jugador por equipo -> combate directo (rol: espada rechazada)
                    continuacion = self.payoff(remA_next[0], remB_next[0], "espada")
                elif k == 4:
                    # quedan 2 -> emparejamiento forzoso:
                    # espada rechazada vs espada rechazada, descarte vs descarte
                    rechazada_A = [x for x in swordsA if x != sa][0]
                    rechazada_B = [x for x in swordsB if x != sb][0]
                    descarte_A = [x for x in remA_next if x != rechazada_A][0]
                    descarte_B = [x for x in remB_next if x != rechazada_B][0]
                    continuacion = (self.payoff(rechazada_A, rechazada_B, "espada")
                                     + self.payoff(descarte_A, descarte_B, "descarte"))
                else:
                    # quedan 3 o más -> se repite el proceso (recursión)
                    continuacion = self.solve_round(remA_next, remB_next)["value"]

                mat[i, j] = inmediato + continuacion

        value, row_strat, col_strat = solve_matrix_game(mat)
        return {
            "value": value,
            "accept_A_strategy": {filas[i]: row_strat[i] for i in range(2)},
            "accept_B_strategy": {cols[j]: col_strat[j] for j in range(2)},
            "matrix": mat,
        }

    # --------------------------------------------------------
    # Etapa 2 (espadas): dado el escudo ya revelado de cada equipo,
    # cada equipo elige en secreto sus 2 espadas.
    # --------------------------------------------------------
    def _swords_game(self, shieldA, shieldB, remA, remB):
        candA = [x for x in remA if x != shieldA]
        candB = [x for x in remB if x != shieldB]
        subsetsA = list(itertools.combinations(candA, 2))
        subsetsB = list(itertools.combinations(candB, 2))

        mat = np.zeros((len(subsetsA), len(subsetsB)))
        detalles = {}
        for i, sA in enumerate(subsetsA):
            for j, sB in enumerate(subsetsB):
                res = self._accept_game(shieldA, shieldB, sA, sB, remA, remB)
                mat[i, j] = res["value"]
                detalles[(sA, sB)] = res

        value, row_strat, col_strat = solve_matrix_game(mat)
        return {
            "value": value,
            "swords_A_strategy": {subsetsA[i]: row_strat[i] for i in range(len(subsetsA))},
            "swords_B_strategy": {subsetsB[j]: col_strat[j] for j in range(len(subsetsB))},
            "detalles": detalles,
            "matrix": mat,
        }

    # --------------------------------------------------------
    # Etapa 1 (escudo): cada equipo elige en secreto su escudo.
    # --------------------------------------------------------
    def solve_round(self, remA, remB):
        remA = tuple(sorted(remA))
        remB = tuple(sorted(remB))
        key = (remA, remB)
        if key in self._memo_round:
            return self._memo_round[key]

        k = len(remA)
        assert k == len(remB), "Ambos equipos deben tener el mismo número de jugadores restantes."
        assert k >= 3, "solve_round solo aplica cuando quedan >= 3 jugadores por equipo."

        mat = np.zeros((k, k))
        detalles = {}
        for i, sA in enumerate(remA):
            for j, sB in enumerate(remB):
                res = self._swords_game(sA, sB, remA, remB)
                mat[i, j] = res["value"]
                detalles[(sA, sB)] = res

        value, row_strat, col_strat = solve_matrix_game(mat)
        resultado = {
            "value": value,
            "shield_A_strategy": {remA[i]: row_strat[i] for i in range(k)},
            "shield_B_strategy": {remB[j]: col_strat[j] for j in range(k)},
            "detalles": detalles,
            "matrix": mat,
            "remA": remA,
            "remB": remB,
        }
        self._memo_round[key] = resultado
        return resultado

    # --------------------------------------------------------
    # UTILIDAD: elegir la opción de mayor probabilidad (moda)
    # --------------------------------------------------------
    @staticmethod
    def _moda(strategy_dict):
        return max(strategy_dict.items(), key=lambda kv: kv[1])

    def _fmt_A(self, idx):
        return self.nombres_A[idx]

    def _fmt_B(self, idx):
        return self.nombres_B[idx]

    # --------------------------------------------------------
    # DISPLAY: recorre el juego ronda a ronda mostrando las
    # estrategias óptimas (escudo, espadas, espada aceptada)
    # --------------------------------------------------------
    def jugar_y_mostrar(self, remA=None, remB=None, ronda=1):
        if remA is None:
            remA = tuple(range(self.N))
        if remB is None:
            remB = tuple(range(self.N))

        k = len(remA)
        print("=" * 70)
        print(f"RONDA {ronda}  (quedan {k} jugadores por equipo)")
        print("=" * 70)

        if k == 1:
            a, b = remA[0], remB[0]
            print(f"Combate final: {self._fmt_A(a)}  vs  {self._fmt_B(b)}")
            print(f"Resultado esperado (rol: espada): {self.payoff(a, b, 'espada'):.3f}")
            return self.payoff(a, b, 'espada')

        if k == 2:
            # No debería llegar aquí directamente salvo llamada manual;
            # se maneja normalmente dentro de _accept_game (caso k==4 anterior).
            print("Quedan 2 jugadores: revisa la ronda anterior, el emparejamiento ya es forzoso.")
            return None

        # --- Etapa 1: escudo ---
        round_res = self.solve_round(remA, remB)
        shieldA_strat = round_res["shield_A_strategy"]
        shieldB_strat = round_res["shield_B_strategy"]
        shieldA_best, pA = self._moda(shieldA_strat)
        shieldB_best, pB = self._moda(shieldB_strat)

        print("\n--- PASO 1-2: ESCUDO ---")
        print(f"Equipo A -> escudo óptimo: {self._fmt_A(shieldA_best)}  (prob. {pA*100:.1f}%)")
        if len(shieldA_strat) > 1:
            otros = {self._fmt_A(k_): f"{v*100:.1f}%" for k_, v in shieldA_strat.items() if v > 1e-6}
            print(f"   Distribución completa: {otros}")
        print(f"Equipo B -> escudo óptimo: {self._fmt_B(shieldB_best)}  (prob. {pB*100:.1f}%)")
        if len(shieldB_strat) > 1:
            otros = {self._fmt_B(k_): f"{v*100:.1f}%" for k_, v in shieldB_strat.items() if v > 1e-6}
            print(f"   Distribución completa: {otros}")

        # --- Etapa 2: espadas (condicionado al escudo más probable) ---
        swords_res = round_res["detalles"][(shieldA_best, shieldB_best)]
        swA_strat = swords_res["swords_A_strategy"]
        swB_strat = swords_res["swords_B_strategy"]
        swA_best, pSA = self._moda(swA_strat)
        swB_best, pSB = self._moda(swB_strat)

        print("\n--- PASO 3-4: ESPADAS ---")
        print(f"Equipo A -> espadas óptimas: "
              f"{[self._fmt_A(x) for x in swA_best]}  (prob. {pSA*100:.1f}%)")
        print(f"Equipo B -> espadas óptimas: "
              f"{[self._fmt_B(x) for x in swB_best]}  (prob. {pSB*100:.1f}%)")

        # --- Etapa 3: aceptación (condicionado a espadas más probables) ---
        accept_res = swords_res["detalles"][(swA_best, swB_best)]
        accA_strat = accept_res["accept_A_strategy"]
        accB_strat = accept_res["accept_B_strategy"]
        acc_sb, pAA = self._moda(accA_strat)   # espada de B que A acepta
        acc_sa, pAB = self._moda(accB_strat)   # espada de A que B acepta

        print("\n--- PASO 5-6: ACEPTACIÓN ---")
        print(f"Equipo A acepta la espada de B: {self._fmt_B(acc_sb)}  (prob. {pAA*100:.1f}%)"
              f"  ->  combate: {self._fmt_A(shieldA_best)} (escudo A) vs {self._fmt_B(acc_sb)} (espada B)")
        print(f"Equipo B acepta la espada de A: {self._fmt_A(acc_sa)}  (prob. {pAB*100:.1f}%)"
              f"  ->  combate: {self._fmt_A(acc_sa)} (espada A) vs {self._fmt_B(shieldB_best)} (escudo B)")

        print(f"\nValor esperado de esta ronda en adelante: {round_res['value']:.3f}")

        # --- Calcular jugadores restantes tras esta ronda ---
        remA_next = tuple(x for x in remA if x not in (shieldA_best, acc_sa))
        remB_next = tuple(x for x in remB if x not in (shieldB_best, acc_sb))

        if len(remA_next) == 1:
            print("\n>> Queda 1 jugador por equipo. Combate final:")
            a, b = remA_next[0], remB_next[0]
            print(f"   {self._fmt_A(a)}  vs  {self._fmt_B(b)}   "
                  f"(resultado esperado, rol espada: {self.payoff(a, b, 'espada'):.3f})")
        elif len(remA_next) == 2:
            rechazada_A = [x for x in swA_best if x != acc_sa][0]
            rechazada_B = [x for x in swB_best if x != acc_sb][0]
            descarte_A = [x for x in remA_next if x != rechazada_A][0]
            descarte_B = [x for x in remB_next if x != rechazada_B][0]
            print("\n>> Quedan 2 jugadores por equipo. Emparejamiento forzoso:")
            print(f"   Espadas rechazadas: {self._fmt_A(rechazada_A)} vs {self._fmt_B(rechazada_B)}  "
                  f"(resultado esperado, rol espada: {self.payoff(rechazada_A, rechazada_B, 'espada'):.3f})")
            print(f"   Descartes:          {self._fmt_A(descarte_A)} vs {self._fmt_B(descarte_B)}  "
                  f"(resultado esperado, rol descarte: {self.payoff(descarte_A, descarte_B, 'descarte'):.3f})")
        else:
            self.jugar_y_mostrar(remA_next, remB_next, ronda=ronda + 1)

        return round_res["value"]


# ------------------------------------------------------------
# 4. EJEMPLO DE USO
# ------------------------------------------------------------
if __name__ == "__main__":
    np.random.seed(42)
    N = 5
    # 3 matrices de payout de ejemplo: sustitúyelas por tus matrices reales.
    # M_X[i, j] = resultado cuando A_i se enfrenta a B_j actuando A_i con rol X
    # (positivo favorece a A). Cada rol puede tener una dinámica distinta.
    M_escudo = np.random.uniform(-1, 1, size=(N, N)).round(2)
    M_espada = np.random.uniform(-1, 1, size=(N, N)).round(2)
    M_descarte = np.random.uniform(-1, 1, size=(N, N)).round(2)

    nombres_A = [f"Jugador_A{i+1}" for i in range(N)]
    nombres_B = [f"Jugador_B{i+1}" for i in range(N)]

    print("Matriz de payout ESCUDO:")
    print(M_escudo)
    print("\nMatriz de payout ESPADA:")
    print(M_espada)
    print("\nMatriz de payout DESCARTE:")
    print(M_descarte)
    print()

    juego = EspadasYEscudo(M_escudo, M_espada, M_descarte, nombres_A, nombres_B)
    valor_total = juego.jugar_y_mostrar()
    print("\n" + "=" * 70)
    print(f"VALOR ESPERADO TOTAL DEL JUEGO (para el equipo A): {valor_total:.3f}")
    print("=" * 70)

Matriz de payout ESCUDO:
[[-0.25  0.9   0.46  0.2  -0.69]
 [-0.69 -0.88  0.73  0.2   0.42]
 [-0.96  0.94  0.66 -0.58 -0.64]
 [-0.63 -0.39  0.05 -0.14 -0.42]
 [ 0.22 -0.72 -0.42 -0.27 -0.09]]

Matriz de payout ESPADA:
[[ 0.57 -0.6   0.03  0.18 -0.91]
 [ 0.22 -0.66 -0.87  0.9   0.93]
 [ 0.62 -0.39 -0.8   0.37 -0.12]
 [-0.76 -0.01 -0.93  0.82 -0.48]
 [ 0.33 -0.38  0.04  0.09 -0.63]]

Matriz de payout DESCARTE:
[[ 0.94  0.55  0.88  0.79  0.2 ]
 [ 0.84 -0.82 -0.61 -0.91 -0.35]
 [-0.22 -0.46  0.66 -0.29 -0.44]
 [ 0.09 -0.72  0.6  -0.85  0.97]
 [ 0.54 -0.6  -0.99  0.63  0.41]]

RONDA 1  (quedan 5 jugadores por equipo)

--- PASO 1-2: ESCUDO ---
Equipo A -> escudo óptimo: Jugador_A2  (prob. 66.4%)
   Distribución completa: {'Jugador_A2': '66.4%', 'Jugador_A4': '33.6%'}
Equipo B -> escudo óptimo: Jugador_B2  (prob. 83.6%)
   Distribución completa: {'Jugador_B2': '83.6%', 'Jugador_B3': '16.4%'}

--- PASO 3-4: ESPADAS ---
Equipo A -> espadas óptimas: ['Jugador_A3', 'Jugador_A4']  (prob. 100.0%)
Eq

In [ ]:
# ============================================================
# ASISTENTE INTERACTIVO - EQUIPO A
# ============================================================
# NOTA: esta celda asume que la clase EspadasYEscudo (y sus funciones
# auxiliares: solve_matrix_game, reduce_dominance, etc.) ya está definida
# en una celda anterior del mismo notebook. No hace falta volver a importarla.

import numpy as np
import pandas as pd
from IPython.display import display

_SIN_PREGUNTAR = object()  # centinela para "aún no se ha preguntado"


class AsistenteEspadasYEscudo:
    """
    Asistente paso a paso para el EQUIPO A. Guía al usuario por las 3 fases
    de cada ronda (escudo -> espadas -> aceptación), mostrando rankings
    simétricos (equipo propio y rival) con valor esperado, spread y
    predicciones, y pidiendo por input() las decisiones reales a medida
    que se van revelando.

    - Pulsar Enter siempre selecciona la opción más probable/recomendada.
    - En cualquier pregunta se puede responder con el NOMBRE del jugador
      o con el NÚMERO de la fila en la que aparece en la tabla mostrada.
    """

    def __init__(self, M_escudo, M_espada, M_descarte, nombres_A=None, nombres_B=None):
        self.motor = EspadasYEscudo(M_escudo, M_espada, M_descarte, nombres_A, nombres_B)

    # --------------------------------------------------------
    # Helpers de formato
    # --------------------------------------------------------
    def _fA(self, i):
        return self.motor._fmt_A(i)

    def _fB(self, j):
        return self.motor._fmt_B(j)

    @staticmethod
    def _moda(d):
        return max(d.items(), key=lambda kv: kv[1])

    @staticmethod
    def _ordenar_por(filas, ids, columna, ascending=False):
        """
        Construye un DataFrame a partir de `filas` y lo ordena por `columna`,
        devolviendo también la lista `ids` reordenada de forma sincronizada
        (para poder mapear "posición en la tabla" -> identificador real).
        """
        df = pd.DataFrame(filas)
        orden = sorted(range(len(ids)), key=lambda i: df.iloc[i][columna], reverse=not ascending)
        df_sorted = df.iloc[orden].reset_index(drop=True)
        ids_sorted = [ids[i] for i in orden]
        return df_sorted, ids_sorted

    @staticmethod
    def _estilo_tabla(df, col_nombre, col_prob):
        """
        Devuelve un pandas Styler que:
        - Colorea la celda `col_nombre` en verde, con intensidad
          proporcional al valor (0-100) de `col_prob`.
        - Redondea columnas numéricas a 2 decimales.
        - Redondea columnas de porcentaje (que contienen "(%)" en el
          nombre) a 2 decimales y añade el símbolo %.
        """
        def _colorear(row):
            p = max(0.0, min(1.0, row[col_prob] / 100.0))
            alpha = 0.12 + 0.75 * p  # entre 12% y 87% de opacidad
            estilos = [''] * len(row)
            idx = row.index.get_loc(col_nombre)
            estilos[idx] = f'background-color: rgba(34,139,34,{alpha:.2f}); font-weight: 600;'
            return estilos

        fmt = {}
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                if "(%)" in col:
                    fmt[col] = "{:.2f}%"
                else:
                    fmt[col] = "{:.2f}"

        return df.style.apply(_colorear, axis=1).format(fmt)

    def _condicionado_escudo(self, round_res, shieldA, shieldB, perspectiva):
        """
        Dado un par de escudos, devuelve la información condicionada de la
        fase de espadas: el par de espadas más probable del RIVAL (visto
        desde `perspectiva`) y cuál de esas 2 deberíamos aceptar nosotros.
        """
        swords_res = round_res["detalles"][(shieldA, shieldB)]
        if perspectiva == 'A':
            top_pair_rival, p_pair = self._moda(swords_res["swords_B_strategy"])
            own_modal_pair, _ = self._moda(swords_res["swords_A_strategy"])
            detalle2 = swords_res["detalles"][(own_modal_pair, top_pair_rival)]
            recomendada, p_rec = self._moda(detalle2["accept_A_strategy"])
        else:
            top_pair_rival, p_pair = self._moda(swords_res["swords_A_strategy"])
            own_modal_pair, _ = self._moda(swords_res["swords_B_strategy"])
            detalle2 = swords_res["detalles"][(top_pair_rival, own_modal_pair)]
            recomendada, p_rec = self._moda(detalle2["accept_B_strategy"])
        return top_pair_rival, p_pair, recomendada, p_rec

    # --------------------------------------------------------
    # Helpers de input (Enter = recomendado; nombre o nº de fila válidos)
    # --------------------------------------------------------
    def _pedir_opcion(self, mensaje, orden_tabla, formatter, default=None):
        """
        Pide UNA opción. `orden_tabla` debe ser la lista de opciones EN EL
        MISMO ORDEN en que se mostraron en la tabla (fila 1 = primer elemento).
        Acepta: Enter (-> default), el nombre exacto, o el nº de fila (1-indexado).
        """
        etiquetas = {formatter(o): o for o in orden_tabla}
        sugerencia = f"  [Enter = {formatter(default)}]" if default is not None else ""
        listado = "\n".join(f"  {idx + 1}. {formatter(o)}" for idx, o in enumerate(orden_tabla))
        while True:
            resp = input(f"{mensaje}{sugerencia}\n(nombre o nº de fila de la tabla)\n{listado}\n> ").strip()
            if resp == "" and default is not None:
                return default
            if resp in etiquetas:
                return etiquetas[resp]
            try:
                pos = int(resp)
                if 1 <= pos <= len(orden_tabla):
                    return orden_tabla[pos - 1]
            except ValueError:
                pass
            print("  -> Entrada no válida, intenta de nuevo.")

    def _pedir_opcion_opcional(self, mensaje, orden_tabla, formatter):
        """Igual que _pedir_opcion pero Enter devuelve None (no se sabe / no aplica)."""
        etiquetas = {formatter(o): o for o in orden_tabla}
        listado = "\n".join(f"  {idx + 1}. {formatter(o)}" for idx, o in enumerate(orden_tabla))
        while True:
            resp = input(f"{mensaje}  [Enter = no lo sé / no aplicar]\n{listado}\n> ").strip()
            if resp == "":
                return None
            if resp in etiquetas:
                return etiquetas[resp]
            try:
                pos = int(resp)
                if 1 <= pos <= len(orden_tabla):
                    return orden_tabla[pos - 1]
            except ValueError:
                pass
            print("  -> Entrada no válida, intenta de nuevo.")

    def _pedir_combo(self, mensaje, orden_tabla_combos, opciones_individuales, formatter, default=None):
        """
        Pide una pareja de jugadores (espadas). Acepta:
        - Enter -> default
        - Un único número = nº de fila en la tabla de combinaciones mostrada
        - Dos nombres separados por coma
        - Dos números separados por coma = posición (1-indexada) dentro de
          `opciones_individuales` (los candidatos disponibles, en el orden
          en que se listan en el propio mensaje)
        """
        sugerencia = f"  [Enter = {formatter(default[0])}, {formatter(default[1])}]" if default is not None else ""
        listado_combos = "\n".join(
            f"  {idx + 1}. {formatter(c[0])}, {formatter(c[1])}" for idx, c in enumerate(orden_tabla_combos))
        listado_individual = ", ".join(f"{i + 1}={formatter(o)}" for i, o in enumerate(opciones_individuales))
        while True:
            resp = input(
                f"{mensaje}{sugerencia}\n"
                f"Puedes escribir el Nº de fila de la tabla de combinaciones:\n{listado_combos}\n"
                f"...o 2 nombres/posiciones individuales separados por coma ({listado_individual})\n> "
            ).strip()
            if resp == "" and default is not None:
                return default

            try:
                pos = int(resp)
                if 1 <= pos <= len(orden_tabla_combos):
                    return orden_tabla_combos[pos - 1]
            except ValueError:
                pass

            partes = [p.strip() for p in resp.replace(";", ",").split(",") if p.strip() != ""]
            if len(partes) == 2:
                etiquetas = {formatter(o): o for o in opciones_individuales}
                elegidos = []
                ok = True
                for p in partes:
                    if p in etiquetas:
                        elegidos.append(etiquetas[p])
                    else:
                        try:
                            val = int(p)
                            if 1 <= val <= len(opciones_individuales):
                                elegidos.append(opciones_individuales[val - 1])
                            else:
                                ok = False
                        except ValueError:
                            ok = False
                if ok and len(set(elegidos)) == 2:
                    return tuple(sorted(elegidos))
            print("  -> Entrada no válida, intenta de nuevo.")

    # --------------------------------------------------------
    # FASE 1: ESCUDO
    # --------------------------------------------------------
    def _tabla_escudo(self, round_res, remA, remB, perspectiva):
        """Construye la tabla de opciones de escudo para 'A' o 'B' (simétrico)."""
        mat = round_res["matrix"]  # filas=remA, columnas=remB (escala M: + favorece a A)

        if perspectiva == 'A':
            own_list, own_strat = remA, round_res["shield_A_strategy"]
            rival_list, rival_strat = remB, round_res["shield_B_strategy"]
            fmt_own, fmt_rival = self._fA, self._fB
        else:
            own_list, own_strat = remB, round_res["shield_B_strategy"]
            rival_list, rival_strat = remA, round_res["shield_A_strategy"]
            fmt_own, fmt_rival = self._fB, self._fA

        rival_modal, p_rival_modal = self._moda(rival_strat)
        rival_vec = np.array([rival_strat[r] for r in rival_list])

        filas, ids = [], []
        for own in own_list:
            if perspectiva == 'A':
                i = remA.index(own)
                valores = mat[i, :]
                shieldA_cond, shieldB_cond = own, rival_modal
            else:
                j = remB.index(own)
                valores = mat[:, j]
                shieldA_cond, shieldB_cond = rival_modal, own

            valor_esperado = float(valores @ rival_vec)
            if perspectiva == 'A':
                peor, mejor = float(valores.min()), float(valores.max())
            else:
                # equipo B minimiza -> su "mejor caso" es el valor M más bajo
                mejor, peor = float(valores.min()), float(valores.max())

            top_pair_rival, p_pair, recomendada, p_rec = self._condicionado_escudo(
                round_res, shieldA_cond, shieldB_cond, perspectiva)

            filas.append({
                "Escudo propio": fmt_own(own),
                "Prob. óptima (%)": own_strat[own] * 100,
                "Valor esperado": valor_esperado,
                "Peor caso": peor,
                "Mejor caso": mejor,
                "Spread": abs(mejor - peor),
                "Escudo rival supuesto": fmt_rival(rival_modal),
                "Espadas más prob. del rival": f"{fmt_rival(top_pair_rival[0])}, {fmt_rival(top_pair_rival[1])}",
                "Prob. jugada rival (%)": p_pair * 100,
                "Deberíamos aceptar": fmt_rival(recomendada),
            })
            ids.append(own)

        df, ids_sorted = self._ordenar_por(filas, ids, "Prob. óptima (%)")
        return self._estilo_tabla(df, "Escudo propio", "Prob. óptima (%)"), ids_sorted

    def _tabla_escudo_conocido(self, round_res, remA, remB, shieldB_conocido):
        """
        Tabla del escudo propio ASUMIENDO que ya se sabe con certeza cuál va
        a ser el escudo rival (shieldB_conocido). No hace falta tabla simétrica
        de B, porque su elección ya no se trata como incierta.
        """
        filas, ids = [], []
        for a in remA:
            swords_res = round_res["detalles"][(a, shieldB_conocido)]
            valor = float(swords_res["value"])
            rival_pair, p_riv = self._moda(swords_res["swords_B_strategy"])
            own_pair, _ = self._moda(swords_res["swords_A_strategy"])
            detalle2 = swords_res["detalles"][(own_pair, rival_pair)]
            recomendada, p_rec = self._moda(detalle2["accept_A_strategy"])
            filas.append({
                "Escudo propio": self._fA(a),
                "Valor esperado (escudo rival conocido)": valor,
                "Espadas más prob. del rival": f"{self._fB(rival_pair[0])}, {self._fB(rival_pair[1])}",
                "Prob. jugada rival (%)": p_riv * 100,
                "Deberíamos aceptar": self._fB(recomendada),
            })
            ids.append(a)

        df, ids_sorted = self._ordenar_por(filas, ids, "Valor esperado (escudo rival conocido)")
        # columna sintética de color (ranking relativo, no es una probabilidad real)
        vals = df["Valor esperado (escudo rival conocido)"].to_numpy(dtype=float)
        rango = vals.max() - vals.min()
        df["Ranking relativo (%)"] = 100.0 if rango < 1e-9 else (vals - vals.min()) / rango * 100.0
        styled = self._estilo_tabla(df, "Escudo propio", "Ranking relativo (%)")
        return styled, ids_sorted

    def _tabla_worst_case(self, round_res, remA, remB):
        """
        Escenario worst-case: el rival "conoce" cuál es nuestro escudo (para
        cada candidato) y responde de forma adversarial (minimiza nuestro
        valor). Muestra qué escudo/espadas lanzaría el rival en ese caso,
        qué deberíamos aceptar, y si nuestro escudo "se sacrifica" o no.
        """
        mat = round_res["matrix"]
        filas, ids = [], []
        for a in remA:
            i = remA.index(a)
            valores = mat[i, :]
            j_worst = int(np.argmin(valores))
            shieldB_worst = remB[j_worst]
            valor_peor = float(valores[j_worst])

            swords_res_w = round_res["detalles"][(a, shieldB_worst)]
            own_pair_w, _ = self._moda(swords_res_w["swords_A_strategy"])
            rival_pair_w, p_riv_w = self._moda(swords_res_w["swords_B_strategy"])
            detalle_w = swords_res_w["detalles"][(own_pair_w, rival_pair_w)]
            recomendada_w, p_rec_w = self._moda(detalle_w["accept_A_strategy"])
            directo_w = self.motor.payoff(a, recomendada_w, "escudo")

            otra_w = [x for x in rival_pair_w if x != recomendada_w]
            otra_w = otra_w[0] if otra_w else None
            if otra_w is not None:
                directo_otra_w = self.motor.payoff(a, otra_w, "escudo")
                sacrificado = directo_otra_w > directo_w + 1e-6
            else:
                directo_otra_w = directo_w
                sacrificado = False

            if sacrificado:
                comentario = (f"Sí: cede {(directo_otra_w - directo_w):.2f} pts individuales "
                              f"aceptando a {self._fB(recomendada_w)} en vez de a {self._fB(otra_w)}, "
                              f"pero mejora el resultado total del equipo.")
            else:
                comentario = "No: la mejor opción individual del escudo también es la mejor para el equipo."

            filas.append({
                "Escudo propio": self._fA(a),
                "Valor esperado (peor caso)": valor_peor,
                "Escudo rival (peor caso)": self._fB(shieldB_worst),
                "Espadas rival (peor caso)": f"{self._fB(rival_pair_w[0])}, {self._fB(rival_pair_w[1])}",
                "Deberíamos aceptar": self._fB(recomendada_w),
                "¿Escudo sacrificado?": comentario,
            })
            ids.append(a)

        df, ids_sorted = self._ordenar_por(filas, ids, "Valor esperado (peor caso)")
        vals = df["Valor esperado (peor caso)"].to_numpy(dtype=float)
        rango = vals.max() - vals.min()
        df["Ranking relativo (%)"] = 100.0 if rango < 1e-9 else (vals - vals.min()) / rango * 100.0
        styled = self._estilo_tabla(df, "Escudo propio", "Ranking relativo (%)")
        return styled, ids_sorted

    def _preguntar_escudo_rival_conocido(self, remB):
        print("\n" + "=" * 90)
        print("¿SABES CON SEGURIDAD CUÁL SERÁ EL ESCUDO INICIAL DEL RIVAL?")
        print("=" * 90)
        print("Si estás muy seguro, indícalo para calcular el mejor escudo propio asumiendo ese dato.")
        print("Si no lo sabes, pulsa Enter y se calculará el equilibrio habitual (sin asumir nada).")
        return self._pedir_opcion_opcional("Escudo rival inicial (si lo conoces):", list(remB), self._fB)

    def _fase_escudo(self, remA, remB, shieldB_conocido=None):
        round_res = self.motor.solve_round(remA, remB)

        print("\n" + "=" * 90)
        print(f"FASE 1 - SELECCIÓN DE ESCUDO   (quedan {len(remA)} jugadores por equipo)")
        print("=" * 90)

        if shieldB_conocido is not None:
            print(f"\n(Calculando asumiendo que el escudo rival será {self._fB(shieldB_conocido)})")
            print("\n-- Equipo A: opciones de escudo (escudo rival asumido conocido) --")
            styled_A, orden_A = self._tabla_escudo_conocido(round_res, remA, remB, shieldB_conocido)
            display(styled_A)
            orden_B = list(remB)
            default_shieldA = orden_A[0]
            default_shieldB = shieldB_conocido
        else:
            print("\n-- Equipo A: opciones de escudo --")
            styled_A, orden_A = self._tabla_escudo(round_res, remA, remB, 'A')
            display(styled_A)

            print("\n-- Equipo B: opciones de escudo (vista simétrica) --")
            styled_B, orden_B = self._tabla_escudo(round_res, remA, remB, 'B')
            display(styled_B)

            default_shieldA = self._moda(round_res["shield_A_strategy"])[0]
            default_shieldB = self._moda(round_res["shield_B_strategy"])[0]

        print("\n-- Escenario worst-case (el rival conoce tu escudo y responde de forma adversarial) --")
        styled_worst, _ = self._tabla_worst_case(round_res, remA, remB)
        display(styled_worst)

        shieldA_real = self._pedir_opcion(
            "\n¿Qué jugador ha seleccionado TU equipo como escudo?",
            orden_A, self._fA, default=default_shieldA)
        shieldB_real = self._pedir_opcion(
            "¿Qué jugador ha revelado el RIVAL como escudo?",
            orden_B, self._fB, default=default_shieldB)

        return shieldA_real, shieldB_real, round_res

    # --------------------------------------------------------
    # FASE 2: ESPADAS
    # --------------------------------------------------------
    def _fase_espadas(self, remA, remB, shieldA_real, shieldB_real):
        swords_res = self.motor._swords_game(shieldA_real, shieldB_real, remA, remB)
        mat = swords_res["matrix"]
        swA_strat = swords_res["swords_A_strategy"]
        swB_strat = swords_res["swords_B_strategy"]
        subsetsA = list(swA_strat.keys())
        subsetsB = list(swB_strat.keys())
        swB_vec = np.array([swB_strat[s] for s in subsetsB])
        swA_vec = np.array([swA_strat[s] for s in subsetsA])

        rival_modal_pair, p_rival_modal = self._moda(swB_strat)
        own_modal_pair, p_own_modal = self._moda(swA_strat)

        # --- Ranking de NUESTRAS combinaciones de espadas ---
        filas_propias, ids_propias = [], []
        for i, sA in enumerate(subsetsA):
            valor_esperado = float(mat[i, :] @ swB_vec)
            peor = float(mat[i, :].min())
            mejor = float(mat[i, :].max())
            detalle = swords_res["detalles"][(sA, rival_modal_pair)]
            sword_aceptada_rival, p_acc = self._moda(detalle["accept_B_strategy"])
            filas_propias.append({
                "Espadas propias": f"{self._fA(sA[0])}, {self._fA(sA[1])}",
                "Prob. óptima (%)": swA_strat[sA] * 100,
                "Valor esperado": valor_esperado,
                "Peor caso": peor,
                "Mejor caso": mejor,
                "Spread": abs(mejor - peor),
                f"Rival probablemente acepta (si lanza {self._fB(rival_modal_pair[0])},{self._fB(rival_modal_pair[1])})":
                    self._fA(sword_aceptada_rival),
                "Prob. (%)": p_acc * 100,
            })
            ids_propias.append(sA)
        df_propias, orden_propias = self._ordenar_por(filas_propias, ids_propias, "Prob. óptima (%)")
        df_propias_styled = self._estilo_tabla(df_propias, "Espadas propias", "Prob. óptima (%)")

        # --- Ranking de espadas PREDICHAS del rival (simétrico) ---
        filas_rival, ids_rival = [], []
        for j, sB in enumerate(subsetsB):
            valor_esperado = float(swA_vec @ mat[:, j])
            # equipo B minimiza -> su "mejor caso" es el valor M más bajo
            mejor, peor = float(mat[:, j].min()), float(mat[:, j].max())
            detalle = swords_res["detalles"][(own_modal_pair, sB)]
            sword_a_aceptar, p_acc = self._moda(detalle["accept_A_strategy"])
            filas_rival.append({
                "Espadas rival (predicción)": f"{self._fB(sB[0])}, {self._fB(sB[1])}",
                "Prob. óptima (%)": swB_strat[sB] * 100,
                "Valor esperado": valor_esperado,
                "Peor caso": peor,
                "Mejor caso": mejor,
                "Spread": abs(mejor - peor),
                f"Deberíamos aceptar (si nosotros lanzamos {self._fA(own_modal_pair[0])},{self._fA(own_modal_pair[1])})":
                    self._fB(sword_a_aceptar),
                "Prob. (%)": p_acc * 100,
            })
            ids_rival.append(sB)
        df_rival, orden_rival = self._ordenar_por(filas_rival, ids_rival, "Prob. óptima (%)")
        df_rival_styled = self._estilo_tabla(df_rival, "Espadas rival (predicción)", "Prob. óptima (%)")

        print("\n" + "=" * 90)
        print("FASE 2 - SELECCIÓN DE ESPADAS")
        print("=" * 90)
        print(f"\nEscudos ya revelados: {self._fA(shieldA_real)} (propio)  vs  {self._fB(shieldB_real)} (rival)")
        print("\n-- Equipo A: ranking de combinaciones de espadas --")
        display(df_propias_styled)
        print("\n-- Equipo B: ranking de combinaciones de espadas (predicción, vista simétrica) --")
        display(df_rival_styled)

        candA = [x for x in remA if x != shieldA_real]
        candB = [x for x in remB if x != shieldB_real]

        propiaA = self._pedir_combo(
            "\n¿Qué 2 jugadores ha elegido TU equipo como espadas?",
            orden_propias, candA, self._fA, default=own_modal_pair)
        propiaB = self._pedir_combo(
            "¿Qué 2 jugadores ha revelado el RIVAL como espadas?",
            orden_rival, candB, self._fB, default=rival_modal_pair)

        return propiaA, propiaB, swords_res

    def _prediccion_representativa(self, remA_x, remB_x):
        """
        Para una continuación con >=3 jugadores por equipo, predice UN combate
        representativo (el más probable en cadena): escudo modal de A vs la
        espada que A aceptaría del rival más probable. Devuelve
        (jugadorA, jugadorB, valor_M, probabilidad_encadenada) o None si no aplica.
        """
        if len(remA_x) < 3:
            return None
        round_res_x = self.motor.solve_round(remA_x, remB_x)
        shA, pA = self._moda(round_res_x["shield_A_strategy"])
        shB, _ = self._moda(round_res_x["shield_B_strategy"])
        swords_res_x = round_res_x["detalles"][(shA, shB)]
        top_rival_pair, p_pair = self._moda(swords_res_x["swords_B_strategy"])
        own_pair, _ = self._moda(swords_res_x["swords_A_strategy"])
        detalle_x = swords_res_x["detalles"][(own_pair, top_rival_pair)]
        acc, p_acc = self._moda(detalle_x["accept_A_strategy"])
        prob_encadenada = pA * p_pair * p_acc
        return shA, acc, self.motor.payoff(shA, acc, "escudo"), float(prob_encadenada)

    def _explicar_procedencia(self, remA, remB, shieldA_real, shieldB_real,
                               swordsA_real, swordsB_real, Q, recomendada, alternativa):
        """
        Explica de dónde sale concretamente la diferencia de puntos entre aceptar
        `recomendada` o `alternativa`: qué emparejamiento futuro cambia.
        """
        j_modal = int(np.argmax(Q))
        sa_modal = swordsA_real[j_modal]
        remA_next = tuple(x for x in remA if x not in (shieldA_real, sa_modal))

        def remB_next_de(sb):
            return tuple(x for x in remB if x not in (shieldB_real, sb))

        remB_next_rec = remB_next_de(recomendada)
        remB_next_alt = remB_next_de(alternativa)
        k_next = len(remA_next)

        if k_next == 1:
            a = remA_next[0]
            b_rec, b_alt = remB_next_rec[0], remB_next_alt[0]
            val_rec = self.motor.payoff(a, b_rec, "espada")
            val_alt = self.motor.payoff(a, b_alt, "espada")
            return (f"Concretamente: aceptando a {self._fB(recomendada)}, el combate final sería "
                    f"{self._fA(a)} vs {self._fB(b_rec)} (puntúa {val_rec:.2f}); aceptando a "
                    f"{self._fB(alternativa)} sería en cambio {self._fA(a)} vs {self._fB(b_alt)} "
                    f"(puntúa {val_alt:.2f}), una diferencia de {(val_rec - val_alt):+.2f} en ese combate.")

        if k_next == 2:
            rechazada_A = [x for x in swordsA_real if x != sa_modal][0]

            def partes_B(sb):
                rechazada_B = [x for x in swordsB_real if x != sb][0]
                rem_next = remB_next_de(sb)
                descarte_B = [x for x in rem_next if x != rechazada_B][0]
                return rechazada_B, descarte_B

            rejB_rec, descB_rec = partes_B(recomendada)
            rejB_alt, descB_alt = partes_B(alternativa)
            val_rec = self.motor.payoff(rechazada_A, rejB_rec, "espada")
            val_alt = self.motor.payoff(rechazada_A, rejB_alt, "espada")
            return (f"Concretamente: aceptando a {self._fB(recomendada)}, tu jugador rechazado "
                    f"{self._fA(rechazada_A)} se enfrenta en el emparejamiento forzoso a "
                    f"{self._fB(rejB_rec)} (puntúa {val_rec:.2f}); aceptando a {self._fB(alternativa)} "
                    f"se enfrentaría en cambio a {self._fB(rejB_alt)} (puntúa {val_alt:.2f}), "
                    f"una diferencia de {(val_rec - val_alt):+.2f} en ese combate.")

        pred_rec = self._prediccion_representativa(remA_next, remB_next_rec)
        pred_alt = self._prediccion_representativa(remA_next, remB_next_alt)
        if pred_rec is None or pred_alt is None:
            return ""
        shA_r, b_r, val_r, p_r = pred_rec
        shA_a, b_a, val_a, p_a = pred_alt
        return (f"Concretamente: aceptando a {self._fB(recomendada)}, es {p_r*100:.2f}% probable que "
                f"{self._fA(shA_r)} acabe enfrentándose a {self._fB(b_r)} (puntuando {val_r:.2f}); "
                f"aceptando a {self._fB(alternativa)}, es {p_a*100:.2f}% probable que ese combate sea en "
                f"cambio contra {self._fB(b_a)} (puntuando {val_a:.2f}), una diferencia de "
                f"{(val_r - val_a):+.2f} en ese emparejamiento.")

    # --------------------------------------------------------
    # FASE 3: ACEPTACIÓN
    # --------------------------------------------------------
    def _tabla_aceptacion(self, detalle, shieldA_real, shieldB_real, swordsA_real, swordsB_real, perspectiva):
        mat = detalle["matrix"]  # filas = swordsB_real (elección de A), columnas = swordsA_real (elección de B)
        accept_A_strat = detalle["accept_A_strategy"]
        accept_B_strat = detalle["accept_B_strategy"]

        if perspectiva == 'A':
            candidatos = list(swordsB_real)
            propio_strat = accept_A_strat
            rival_vec = np.array([accept_B_strat[sa] for sa in swordsA_real])
            propio_shield = shieldA_real
            fmt_candidato = self._fB
        else:
            candidatos = list(swordsA_real)
            propio_strat = accept_B_strat
            rival_vec = np.array([accept_A_strat[sb] for sb in swordsB_real])
            propio_shield = shieldB_real
            fmt_candidato = self._fA
        col_nombre = "Espada candidata (rival)"

        if perspectiva == 'A':
            constante = float(sum(
                rival_vec[j] * self.motor.payoff(swordsA_real[j], shieldB_real, "espada")
                for j in range(len(swordsA_real))
            ))
        else:
            constante = float(sum(
                rival_vec[i] * self.motor.payoff(shieldA_real, swordsB_real[i], "escudo")
                for i in range(len(swordsB_real))
            ))

        filas, ids = [], []
        for idx, cand in enumerate(candidatos):
            if perspectiva == 'A':
                directo = self.motor.payoff(propio_shield, cand, "escudo")
                total = float(mat[idx, :] @ rival_vec)
            else:
                directo = self.motor.payoff(cand, propio_shield, "espada")
                total = float(mat[:, idx] @ rival_vec)
            continuacion = total - directo - constante
            filas.append({
                col_nombre: fmt_candidato(cand),
                "Prob. óptima (%)": propio_strat[cand] * 100,
                "Valor directo del escudo": directo,
                "Continuación esperada (resto del bracket)": continuacion,
                "Valor esperado total del equipo": total,
            })
            ids.append(cand)
        df, ids_sorted = self._ordenar_por(filas, ids, "Prob. óptima (%)")
        return self._estilo_tabla(df, col_nombre, "Prob. óptima (%)"), ids_sorted

    def _fase_aceptacion(self, remA, remB, swords_res, shieldA_real, shieldB_real, swordsA_real, swordsB_real):
        detalle = swords_res["detalles"][(swordsA_real, swordsB_real)]
        mat = detalle["matrix"]
        accept_A_strat = detalle["accept_A_strategy"]
        accept_B_strat = detalle["accept_B_strategy"]
        recomendada, p_rec = self._moda(accept_A_strat)

        Q = np.array([accept_B_strat[sa] for sa in swordsA_real])
        constante_otro_combate = float(sum(
            Q[j] * self.motor.payoff(swordsA_real[j], shieldB_real, "espada") for j in range(len(swordsA_real))
        ))

        print("\n" + "=" * 90)
        print("FASE 3 - ACEPTACIÓN DE ESPADA")
        print("=" * 90)
        print(f"\nEspadas reveladas -> Propias: {self._fA(swordsA_real[0])}, {self._fA(swordsA_real[1])}"
              f"   |   Rival: {self._fB(swordsB_real[0])}, {self._fB(swordsB_real[1])}")

        print("\n-- Equipo A: qué espada rival debería aceptar tu escudo --")
        styled_A, orden_A = self._tabla_aceptacion(
            detalle, shieldA_real, shieldB_real, swordsA_real, swordsB_real, 'A')
        display(styled_A)
        print("\n-- Equipo B: qué espada propia debería aceptar su escudo (vista simétrica) --")
        styled_B, orden_B = self._tabla_aceptacion(
            detalle, shieldA_real, shieldB_real, swordsA_real, swordsB_real, 'B')
        display(styled_B)

        print(f"\nRECOMENDACIÓN: tu escudo ({self._fA(shieldA_real)}) debería aceptar la espada "
              f"{self._fB(recomendada)}  (prob. óptima: {p_rec*100:.2f}%)")
        print(f"Valor esperado de todo el equipo jugando así: {detalle['value']:.2f}")
        print(f"Valor bruto (matriz ESCUDO) de ese enfrentamiento concreto "
              f"({self._fA(shieldA_real)} vs {self._fB(recomendada)}): "
              f"{self.motor.payoff(shieldA_real, recomendada, 'escudo'):.2f}")

        # --- Justificación si la recomendación NO es la mejor opción individual para el escudo ---
        directo_recomendada = self.motor.payoff(shieldA_real, recomendada, "escudo")
        mejor_directo_sb, mejor_directo_valor = max(
            ((sb, self.motor.payoff(shieldA_real, sb, "escudo")) for sb in swordsB_real),
            key=lambda t: t[1]
        )
        if mejor_directo_sb != recomendada:
            idx_rec = swordsB_real.index(recomendada)
            idx_alt = swordsB_real.index(mejor_directo_sb)
            total_recomendada = float(mat[idx_rec, :] @ Q)
            total_alternativa = float(mat[idx_alt, :] @ Q)
            continuacion_rec = total_recomendada - directo_recomendada - constante_otro_combate
            continuacion_alt = total_alternativa - mejor_directo_valor - constante_otro_combate
            diff = total_recomendada - total_alternativa

            if diff > 1e-6:
                comparativa = (f"es mayor: {total_recomendada:.2f} frente a {total_alternativa:.2f} "
                                f"si se aceptara a {self._fB(mejor_directo_sb)}")
            else:
                comparativa = (f"es igual en ambos casos ({total_recomendada:.2f}): "
                                f"aceptar a {self._fB(recomendada)} es al menos tan buena elección "
                                f"como aceptar a {self._fB(mejor_directo_sb)}")

            procedencia = self._explicar_procedencia(
                remA, remB, shieldA_real, shieldB_real, swordsA_real, swordsB_real,
                Q, recomendada, mejor_directo_sb)

            print(f"\n⚠️  Nota: individualmente tu escudo ({self._fA(shieldA_real)}) sacaría más puntos "
                  f"aceptando a {self._fB(mejor_directo_sb)} ({mejor_directo_valor:.2f} puntos) que aceptando "
                  f"a {self._fB(recomendada)} ({directo_recomendada:.2f} puntos).\n"
                  f"   Aun así, se recomienda aceptar a {self._fB(recomendada)} porque el valor esperado TOTAL "
                  f"del equipo {comparativa}.\n"
                  f"   La diferencia viene del resto de emparejamientos (quién queda disponible para las "
                  f"siguientes rondas): la continuación esperada es {continuacion_rec:.2f} aceptando a "
                  f"{self._fB(recomendada)} frente a {continuacion_alt:.2f} aceptando a "
                  f"{self._fB(mejor_directo_sb)} (diferencia de {(continuacion_rec - continuacion_alt):.2f} "
                  f"puntos a favor de {self._fB(recomendada)}).\n"
                  f"   {procedencia}")

        return detalle, orden_A, orden_B

    # --------------------------------------------------------
    # ORQUESTADOR DE UNA RONDA COMPLETA + RESOLUCIÓN FINAL
    # --------------------------------------------------------
    def jugar_ronda(self, remA=None, remB=None, ronda=1, historial=None, shieldB_conocido=_SIN_PREGUNTAR):
        if remA is None:
            remA = tuple(range(self.motor.N))
        if remB is None:
            remB = tuple(range(self.motor.N))
        if historial is None:
            historial = []

        k = len(remA)
        assert k == len(remB)

        if k == 1:
            a, b = remA[0], remB[0]
            print("\n" + "#" * 90)
            print("COMBATE FINAL (1 jugador por equipo)")
            print("#" * 90)
            valor = self.motor.payoff(a, b, "espada")
            historial.append({"Ronda": f"Final (ronda {ronda})", "Emparejamiento": f"{self._fA(a)} vs {self._fB(b)}",
                               "Rol": "Descarte", "Valor": valor})
            df = pd.DataFrame([historial[-1]])
            display(df.style.format({"Valor": "{:.2f}"}))
            self._mostrar_historial_completo(historial)
            return

        if ronda == 1 and shieldB_conocido is _SIN_PREGUNTAR:
            shieldB_conocido = self._preguntar_escudo_rival_conocido(remB)

        print("\n" + "#" * 90)
        print(f"RONDA {ronda}  -  quedan {k} jugadores por equipo")
        print("#" * 90)

        shieldA_real, shieldB_real, round_res = self._fase_escudo(
            remA, remB, shieldB_conocido if ronda == 1 else None)
        swordsA_real, swordsB_real, swords_res = self._fase_espadas(remA, remB, shieldA_real, shieldB_real)
        detalle_accept, orden_accept_A, orden_accept_B = self._fase_aceptacion(
            remA, remB, swords_res, shieldA_real, shieldB_real, swordsA_real, swordsB_real)

        default_sb = self._moda(detalle_accept["accept_A_strategy"])[0]
        default_sa = self._moda(detalle_accept["accept_B_strategy"])[0]

        sb_aceptada_por_nosotros = self._pedir_opcion(
            "\n¿Qué espada del RIVAL ha aceptado finalmente tu escudo (para enfrentarse a él)?",
            orden_accept_A, self._fB, default=default_sb)
        sa_aceptada_por_rival = self._pedir_opcion(
            "¿Qué espada de TU equipo ha aceptado el escudo rival?",
            orden_accept_B, self._fA, default=default_sa)

        match1 = (shieldA_real, sb_aceptada_por_nosotros)   # nuestro escudo vs espada rival aceptada
        match2 = (sa_aceptada_por_rival, shieldB_real)       # nuestra espada aceptada vs escudo rival

        remA_next = tuple(x for x in remA if x not in (shieldA_real, sa_aceptada_por_rival))
        remB_next = tuple(x for x in remB if x not in (shieldB_real, sb_aceptada_por_nosotros))

        historial.append({"Ronda": ronda,
                           "Emparejamiento": f"{self._fA(match1[0])} vs {self._fB(match1[1])}",
                           "Rol": "Escudo",
                           "Valor": self.motor.payoff(match1[0], match1[1], "escudo")})
        historial.append({"Ronda": ronda,
                           "Emparejamiento": f"{self._fA(match2[0])} vs {self._fB(match2[1])}",
                           "Rol": "Espada",
                           "Valor": self.motor.payoff(match2[0], match2[1], "espada")})

        print("\n--- RESULTADO DE ESTA RONDA ---")
        df_ronda = pd.DataFrame(historial[-2:])
        display(df_ronda.style.format({"Valor": "{:.2f}"}))

        if len(remA_next) == 0:
            print("\nEl juego ha terminado, no quedan jugadores sin emparejar.")
            self._mostrar_historial_completo(historial)
            return

        if len(remA_next) == 1:
            self.jugar_ronda(remA_next, remB_next, ronda=ronda + 1, historial=historial, shieldB_conocido=None)
            return

        if len(remA_next) == 2:
            rechazada_A = [x for x in swordsA_real if x != sa_aceptada_por_rival][0]
            rechazada_B = [x for x in swordsB_real if x != sb_aceptada_por_nosotros][0]
            descarte_A = [x for x in remA_next if x != rechazada_A][0]
            descarte_B = [x for x in remB_next if x != rechazada_B][0]

            print("\n" + "#" * 90)
            print("EMPAREJAMIENTOS FINALES AUTOMÁTICOS (quedaban 2 jugadores por equipo)")
            print("#" * 90)
            historial.append({"Ronda": f"Final (ronda {ronda + 1})",
                               "Emparejamiento": f"{self._fA(rechazada_A)} vs {self._fB(rechazada_B)}",
                               "Rol": "Espada rechazada",
                               "Valor": self.motor.payoff(rechazada_A, rechazada_B, "espada")})
            historial.append({"Ronda": f"Final (ronda {ronda + 1})",
                               "Emparejamiento": f"{self._fA(descarte_A)} vs {self._fB(descarte_B)}",
                               "Rol": "Descarte",
                               "Valor": self.motor.payoff(descarte_A, descarte_B, "descarte")})
            df_final = pd.DataFrame(historial[-2:])
            display(df_final.style.format({"Valor": "{:.2f}"}))
            self._mostrar_historial_completo(historial)
            return

        # quedan 3 o más -> nueva ronda
        self.jugar_ronda(remA_next, remB_next, ronda=ronda + 1, historial=historial, shieldB_conocido=None)

    def _mostrar_historial_completo(self, historial):
        print("\n" + "#" * 90)
        print("MATRIZ COMPLETA DE TODOS LOS EMPAREJAMIENTOS DEL JUEGO")
        print("#" * 90)
        df = pd.DataFrame(historial)
        display(df.style.format({"Valor": "{:.2f}"}))
        total = df["Valor"].sum()
        print(f"\nVALOR TOTAL REAL ACUMULADO DE TODOS LOS ENFRENTAMIENTOS: {total:.2f}")


M_escudo, M_espada, M_descarte, nombres_A, nombres_B = cargar_matrices_desde_sheets("input Wahammer pairing")
# ------------------------------------------------------------
# EJEMPLO DE USO (en Colab, ejecutar y responder a los input())
# ------------------------------------------------------------
if __name__ == "__main__":
    nombres_A = ["Necron", "Votan", "Marine", "Astra", "DG"]
    nombres_B = ["Chaos Demons", "Emperors Children", "Tau", "Deathguard", "Astra"]
    M_escudo = M_escudo
    M_espada = M_espada     # ejemplo: ser espada penaliza ligeramente
    M_descarte = M_descarte   # ejemplo: ser descarte penaliza más

    asistente = AsistenteEspadasYEscudo(M_escudo, M_espada, M_descarte, nombres_A, nombres_B)
    asistente.jugar_ronda()

Nombre de la pestaña a usar [Enter = la primera]: 

Leyendo datos de 'input Wahammer pairing' / pestaña 'Hoja 2'...

Matrices detectadas correctamente en input Wahammer pairing / Hoja 2:
  - escudo   : esquina en fila 9, columna 1, tamaño 4x4
  - espada   : esquina en fila 16, columna 1, tamaño 4x4
  - descarte : esquina en fila 2, columna 1, tamaño 4x4

Jugadores equipo A (filas):    ['Necron', 'Votan', 'Marine', 'Astra']
Jugadores equipo B (columnas): ['Chaos Demons', 'Emperors Children', 'Tau', 'Deathguard']

Modo de combinación: aditivo

M_descarte (base):
[[7. 7. 4. 1.]
 [5. 6. 6. 4.]
 [4. 3. 5. 4.]
 [5. 3. 3. 4.]]

M_escudo (final = descarte + escudo):
[[8. 8. 5. 2.]
 [6. 7. 7. 5.]
 [5. 4. 6. 5.]
 [6. 4. 4. 5.]]

M_espada (final = descarte + espada):
[[6. 6. 3. 0.]
 [4. 5. 5. 3.]
 [3. 2. 4. 3.]
 [4. 2. 2. 3.]]

¿SABES CON SEGURIDAD CUÁL SERÁ EL ESCUDO INICIAL DEL RIVAL?
Si estás muy seguro, indícalo para calcular el mejor escudo propio asumiendo ese dato.
Si no lo sabes, pulsa En

,Escudo propio,Prob. óptima (%),Valor esperado,Peor caso,Mejor caso,Spread,Escudo rival supuesto,Espadas más prob. del rival,Prob. jugada rival (%),Deberíamos aceptar
0,Marine,55.00%,16.65,15.00,18.00,3.00,Emperors Children,"Chaos Demons, Deathguard",75.00%,Deathguard
1,Necron,45.00%,16.65,15.00,18.67,3.67,Emperors Children,"Tau, Deathguard",100.00%,Tau
2,Votan,0.00%,16.65,16.12,17.00,0.88,Emperors Children,"Tau, Deathguard",48.00%,Tau
3,Astra,0.00%,16.61,15.50,17.52,2.02,Emperors Children,"Tau, Deathguard",70.37%,Deathguard



-- Equipo B: opciones de escudo (vista simétrica) --


,Escudo propio,Prob. óptima (%),Valor esperado,Peor caso,Mejor caso,Spread,Escudo rival supuesto,Espadas más prob. del rival,Prob. jugada rival (%),Deberíamos aceptar
0,Emperors Children,55.00%,16.65,18.00,15.00,3.00,Marine,"Necron, Votan",100.00%,Necron
1,Deathguard,45.00%,16.65,18.67,15.00,3.67,Marine,"Votan, Astra",100.00%,Votan
2,Chaos Demons,0.00%,17.55,18.00,16.67,1.33,Marine,"Necron, Astra",100.00%,Necron
3,Tau,0.00%,17.23,18.50,16.12,2.38,Marine,"Necron, Votan",60.00%,Necron



-- Escenario worst-case (el rival conoce tu escudo y responde de forma adversarial) --


...


##########################################################################################
EMPAREJAMIENTOS FINALES AUTOMÁTICOS (quedaban 2 jugadores por equipo)
##########################################################################################


,Ronda,Emparejamiento,Rol,Valor
0,Final (ronda 2),Votan vs Chaos Demons,Espada rechazada,4.00
1,Final (ronda 2),Astra vs Tau,Descarte,3.00



##########################################################################################
MATRIZ COMPLETA DE TODOS LOS EMPAREJAMIENTOS DEL JUEGO
##########################################################################################


,Ronda,Emparejamiento,Rol,Valor
0,1,Marine vs Deathguard,Escudo,5.00
1,1,Necron vs Emperors Children,Espada,6.00
2,Final (ronda 2),Votan vs Chaos Demons,Espada rechazada,4.00
3,Final (ronda 2),Astra vs Tau,Descarte,3.00



VALOR TOTAL REAL ACUMULADO DE TODOS LOS ENFRENTAMIENTOS: 18.00
